# ⚠️ Sabotaged Feature Engineering — Find the Three Leaks
**Lab 3 companion.** A well-meaning colleague "improved" the Manafeth churn workflow below and the validation PR-AUC jumped. **Three of their changes are leaks.** Find each one, write the one-line fix in Lab 3's table, and say which of the three numbers below you would believe.

*Do not fix this notebook — it is the exhibit. Fix your own.*


In [ ]:
# --- Course setup (identical in every lab) -----------------------------------
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_repo_root() -> Path:
    """Locate the course repo root by looking for data/manafeth_customers.parquet."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "data" / "manafeth_customers.parquet").exists():
            return cand
    raise FileNotFoundError(
        "Course data not found — copy the Manafeth data package files into <repo>/data/")

ROOT = find_repo_root()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score

customers = pd.read_parquet(DATA / "manafeth_customers.parquet")

# "I cleaned the data first so every step downstream gets clean input" — colleague
num_cols = ["orders_per_month", "avg_basket_sar", "days_since_last_order",
            "tenure_months", "distinct_categories", "promo_usage_rate", "avg_rating",
            "next_month_orders"]          # "kept the strongest feature from my EDA"

Xn = customers[num_cols].copy()
Xn["avg_rating"] = Xn["avg_rating"].fillna(Xn["avg_rating"].mean())   # global mean
scaler = StandardScaler().fit(Xn)                                     # fit on ALL rows
Xs = pd.DataFrame(scaler.transform(Xn), columns=num_cols)

y = customers["churned_30d"]
X_tr, X_val, y_tr, y_val = train_test_split(Xs, y, test_size=0.25,
                                            stratify=y, random_state=42)
clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42).fit(X_tr, y_tr)
print("validation PR-AUC:", round(average_precision_score(
    y_val, clf.predict_proba(X_val)[:, 1]), 3), " <- would you sign off on this number?")

In [ ]:
# "city looked predictive, so I encoded it with the churn rate per city" — colleague
city_means = customers.groupby("city")["churned_30d"].mean()          # computed on ALL data
Xs2 = Xs.assign(city_risk=customers["city"].map(city_means).values)

scores = cross_val_score(LogisticRegression(class_weight="balanced", max_iter=1000,
                                            random_state=42),
                         Xs2, y, cv=5, scoring="average_precision")
print("5-fold CV PR-AUC with city_risk:", scores.mean().round(3))

## Your findings

Three leaks are planted above. For each: name it, explain the mechanism in one sentence, and give the one-line fix.

<details><summary>Instructor reveal (open after the hunt)</summary>

1. **Stale feature list — `next_month_orders` is the target in disguise.** Churners have 0 by construction; the timestamp test fails it. *Fix: drop the three planted leak columns before modelling.* This is why the PR-AUC above is fiction (~0.88).
2. **Preprocessing fit before the split.** The imputer's global mean and the scaler's statistics were learned from ALL rows, so validation information contaminated training. Small effect here, compounding effect in production. *Fix: fit transforms inside a `Pipeline` so they see training folds only.*
3. **Naive target encoding under CV.** `city_risk` was computed from the full dataset, then "validated" by CV — every fold's validation rows contributed to the encoding it is judged on. *Fix: fit the encoding inside each fold (put it in the pipeline).*

The tell in every case: **the number improved without new information arriving.**
</details>
